In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb

*Load specific modules used here:*

In [2]:
os.chdir(d['py'])
import mProduction
from gmsPython import nestingTree
from valueShares import InitTree, nestedShares

# Production modules

*Load test data:*

In [3]:
t0 = 2019
name = 'TestModelData' #
db = GpyDB(os.path.join(d['data'], name), name = 'test', ws = d['work'])
ws = db.ws

For variables that are defined over $t$, but where we do not yet have an initial value for all $t$, extrapolate from data:

*Note: This forces extrapolation of all variables defined over $t$ - if it is important that some variables are not extrapolated, they should be removed from this statement.*

In [4]:
[symbol.__setattr__('vals', extrapolateUpper(symbol.vals, db('tE')[0])) for symbol in [db[k] for k in db.varDom('t')['t']]];

Subset data to only include production sectors, but keep a full copy:

In [5]:
db0 = db.copy()
AggDB.subsetDB(db, db('s_p'))

## 1. ```StaticNCES```

### 1.1. Default implementation

In [6]:
db_StaticNCES = db.copy()
db_i = db_StaticNCES

Remove durables (and their value from supply side):

In [7]:
db_i['qS'] = db_i('qS')-(adj.rc_pd(db_i('qD'), db_i('dur_p')) * db_i('pD_dur')).groupby(['t','s']).sum()
db_i['qD'] = adj.rc_pd(db_i('qD'), ('not', db_i('dur_p')))
db_i['nestProduction'] = adj.rc_pd(db_i('nestProduction'), ('not', db_i('dur_p').rename('nn')))

*Set up nesting structure from nesting tree:*

In [8]:
nest = nestingTree.Tree('P', tree = db_i('nestProduction').to_list(), f = 'CES_scaled')
nestAgg = nestingTree.AggTree(name = 'P', trees = {t.name: t for t in [nest]}, ws = ws)
nestAgg(namespace = {str(n)+'_input':n for n in db('n')})

*Get initial values of $\mu$-parameters and intermediate goods:*

In [9]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db_i('sigma'), v.db('s'))
vals = v(dbIO = db_i)

*Add to main database:*

In [10]:
[db_i.aom(stdSort(adjMultiIndex.bc(vals[k], db_i('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db_i.aom(vals['mu'], name = 'mu', priority = 'second')
db_i.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

Initialize model instance:

In [11]:
m_StaticNCES = mProduction.getStaticNCES(nestAgg, partial = True, initFromGms = 'initFirmValueBlock')
m_i = m_StaticNCES

Add data and initialize other stuff with default options:

In [12]:
m_i.initStuff(db = db_i)

Solve calibration with robust relaxation method:

In [13]:
solve_StaticNCES_Calib = m_i.solve(state='C') 
# solve_StaticNCES_Calib = m_i.jSolve(10, state = 'C')
solve_i = solve_StaticNCES_Calib

Add solution to main database, remove init methods (we want to use this solution as initial value):

In [14]:
[m_i.db.__setitem__(k, solve_i[k]) for k in m_i.db.getTypes(['var']) if k in solve_i.symbols]; # use solution database
m_i.db.mergeInternal()
m_i.initFromGms = None # no longer use gams code to initialize variables

Test baseline state:

In [15]:
solve_StaticNCES = m_i.solve(state = 'B') # solve baseline state
solve_i = solve_StaticNCES

### 1.2. Simple emission extension:

Initialize emission extension:

In [16]:
# m_StaticNCES_emission = mProduction.StaticNCES_emission(nestAgg, partial = True, abateCosts = False) # Initialize directly from extension class
m_StaticNCES_emission = mProduction.getStaticNCES(nestAgg, extension = 'emission', partial = True, abateCosts = False) # ignore abatement costs
m_i = m_StaticNCES_emission

Test run this as well:

In [17]:
m_i.initStuff(db = db_i) # don't need to add more data, we already rely on the solution from the basic model version here.
solve_StaticNCES_emission = m_i.solve()

Now, test run with some type of abatement costs:

In [18]:
m_i.addAbatementCosts("SqrAdjCosts")
m_i.initStuff() # update group definitions and such
solve_StaticNCES_emissionWithAbate = m_i.solve()

### 1.3. Investment module:

Return to full database and create a copy only with investment sectors:

In [19]:
db_invest = db0.copy()
AggDB.subsetDB(db_invest, db_invest('s_i'))
db_i = db_invest

*Set up nesting structure from nesting tree:*

In [20]:
nest = nestingTree.Tree('I', tree = db_i('nestInvestment').to_list(), f = 'CES_scaled')
nestAgg = nestingTree.AggTree(name = 'I', trees = {t.name: t for t in [nest]}, ws = ws)
nestAgg(namespace = {str(n)+'_input':n for n in db('n')})

*Get initial values of $\mu$-parameters and intermediate goods:*

In [21]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db_i('sigma'), v.db('s'))
vals = v(dbIO = db_i)

*Add to main database:*

In [22]:
[db_i.aom(stdSort(adjMultiIndex.bc(vals[k], db_i('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db_i.aom(vals['mu'], name = 'mu', priority = 'second')
db_i.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

Initialize model instance:

In [23]:
m_invest = mProduction.InvestNCES(nestAgg, partial = True)
m_i = m_invest

Add data and initialize other stuff with default options:

In [24]:
m_i.initStuff(db = db_i)

Solve calibration with robust relaxation method:

In [25]:
solve_investCalib = m_i.solve(state = 'C')
# solve_investCalib = m_i.jSolve(10, state = 'C')
solve_i = solve_investCalib

## 2. ```DynamicNCES```

The dynamic classes currently only contains the same extensions as the static models. Technically, extensions are caught by the parent class, thus both instances here are simple ```DynamicNCES_base```.

### 2.1. Without emissions (default)

In [26]:
db_DynamicNCES = db.copy()
db_i = db_DynamicNCES

*Set up nesting structure from nesting tree:*

In [27]:
nest = nestingTree.Tree('P', tree = db_i('nestProduction').to_list(), f = 'CES_scaled')
nestAgg = nestingTree.AggTree(name = 'P', trees = {t.name: t for t in [nest]}, ws = ws)
nestAgg(namespace = {str(n)+'_input':n for n in db('n')})

Use static user cost as initial guess for price on durables (if no other has been provided):

In [28]:
db_i.aom(db('pD_dur'), name = 'pD', priority = 'first') # pD_dur includes an estimate of the long-run user-cost of capital

*Get initial values of $\mu$-parameters and intermediate goods:*

In [29]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db_i('sigma'), v.db('s'))
vals = v(dbIO = db_i)

*Add to main database:*

In [30]:
[db_i.aom(stdSort(adjMultiIndex.bc(vals[k], db_i('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db_i.aom(vals['mu'], name = 'mu', priority = 'second')
db_i.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

Initialize model instance:

In [31]:
m_DynamicNCES = mProduction.getDynamicNCES(nestAgg, partial = True, initFromGms = 'initFirmValueBlock')
m_i = m_DynamicNCES

Add data and initialize other stuff with default options:

In [32]:
m_i.initStuff(db = db_i)

Solve calibration with robust relaxation method:

In [33]:
solve_DynamicNCES_Calib = m_i.solve(state = 'C')
# solve_DynamicNCES_Calib = m_i.jSolve(10, state = 'C')
solve_i = solve_DynamicNCES_Calib

Add solution to main database, remove init methods (we want to use this solution as initial value):

In [34]:
[m_i.db.__setitem__(k, solve_i[k]) for k in m_i.db.getTypes(['var']) if k in solve_i.symbols]; # use solution database
m_i.db.mergeInternal()
m_i.initFromGms = None # no longer use gams code to initialize variables

Test baseline state:

In [35]:
solve_DynamicNCES = m_i.solve(state = 'B') # solve baseline state
solve_i = solve_DynamicNCES

### 2.2. Emissions

In [36]:
db_DynamicNCES_em = db.copy()
db_i = db_DynamicNCES_em

*Set up nesting structure from nesting tree:*

In [37]:
nest = nestingTree.Tree('P', tree = db_i('nestProduction').to_list(), f = 'CES_scaled')
nestAgg = nestingTree.AggTree(name = 'P', trees = {t.name: t for t in [nest]}, ws = ws)
nestAgg(namespace = {str(n)+'_input':n for n in db('n')})

Use static user cost as initial guess for price on durables (if no other has been provided):

In [38]:
db_i.aom(db('pD_dur'), name = 'pD', priority = 'first') # pD_dur includes an estimate of the long-run user-cost of capital

*Get initial values of $\mu$-parameters and intermediate goods:*

In [39]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db_i('sigma'), v.db('s'))
vals = v(dbIO = db_i)

*Add to main database:*

In [40]:
[db_i.aom(stdSort(adjMultiIndex.bc(vals[k], db_i('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db_i.aom(vals['mu'], name = 'mu', priority = 'second')
db_i.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

With emissions, the module is the same, but we specify that the static version uses the extension 'emission':

In [41]:
m_DynamicNCES_emission = mProduction.getDynamicNCES(nestAgg, extension = 'emission', partial = True)
m_i = m_DynamicNCES_emission

Test run this as well:

In [42]:
m_i.initStuff(db = db_i) # don't need to add more data, we already rely on the solution from the basic model version here.
solve_DynamicNCES_emission = m_i.solve(state = 'C')
# solve_DynamicNCES_emission = m_i.jSolve(10, state = 'C')

Now, test run with some type of abatement costs:

In [43]:
m_i.addAbatementCosts("SqrAdjCosts")
m_i.initStuff(addDurables = False)
solve_DynamicNCES_emissionWithAbate = m_i.solve(state = 'C')
# solve_DynamicNCES_emissionWithAbate = m_i.jSolve(10, state = 'C')

### 2.3. Target profits by calibration of share parameters on durables

Assume AR(1) structure on durables

In [44]:
db_DynamicNCES_TP = db.copy()
db_i = db_DynamicNCES_TP

*Set up nesting structure from nesting tree:*

In [45]:
nest = nestingTree.Tree('P', tree = db_i('nestProduction').to_list(), f = 'CES_adjDur_scaled')
nestAgg = nestingTree.AggTree(name = 'P', trees = {t.name: t for t in [nest]}, ws = ws)
nestAgg(namespace = {str(n)+'_input':n for n in db('n')})

Use static user cost as initial guess for price on durables (if no other has been provided):

In [46]:
db_i.aom(db('pD_dur'), name = 'pD', priority = 'first') # pD_dur includes an estimate of the long-run user-cost of capital

*Get initial values of $\mu$-parameters and intermediate goods:*

In [47]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db_i('sigma'), v.db('s'))
vals = v(dbIO = db_i)

*Add to main database:*

In [48]:
[db_i.aom(stdSort(adjMultiIndex.bc(vals[k], db_i('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db_i.aom(vals['mu'], name = 'mu', priority = 'second')
db_i.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

With emissions, the module is the same, but we specify that the static version uses the extension 'emission':

In [49]:
m_DynamicNCES_TP = mProduction.getDynamicNCES(nestAgg, extension = ['emission','TP'], partial = True)
m_i = m_DynamicNCES_TP

Calibrate and stuff:

In [50]:
m_i.initStuff(db = db_i) # don't need to add more data, we already rely on the solution from the basic model version here.
solve_DynamicNCES_TP = m_i.solve(state = 'C')
# solve_DynamicNCES_emission = m_i.jSolve(10, state = 'C')